# Controlled train-window experiment — Pilot Model v5

Pilot v5: tuned XGBoost с `SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES`, где weather feature — только `temperature_bucket`; weighted D с half-life 180 дней; `floor(max(raw, 0) + 0.5)` и ноль для route 5. Единица прогноза — `(route, date, hour)`, target — `boardings`.

## Изменяемый фактор и временная проверка

Меняется только длина train history: FULL, последние 8, 6, 4 или 2 полных календарных месяца, включая последний месяц train. Удаление старых строк исключает их из fit; recency weighting лишь снижает их влияние. Окно определяется внутри каждого фолда из его train, чтобы validation не задавала cutoff. После обрезки weighted D пересчитывается по оставшимся строкам и `max(sliced_train.date)`. Фолды: Jan–Apr → May–Jun, Jan–Jun → Jul–Aug, Jan–Aug → Sep–Oct. Nov–Dec leaderboard не участвует в выборе окна.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / 'ml/src/train_window_experiment.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Project root not found')
sys.path.insert(0, str(ROOT / 'ml/src'))
import train_window_experiment as experiment

result = experiment.run_experiment()
if not result['full_reference_reproduced']:
    raise ValueError('FULL reference was not reproduced')
print('FULL reference reproduced:', result['full_reference_reproduced'])
print('Classification:', result['classification'])

FULL reference reproduced: True
Classification: TRAIN_WINDOW_REJECTED


## Результаты

Основной критерий — pooled WAPE всех трёх фолдов. Ранние фолды содержат окна, совпадающие с FULL; это видно в таблице train diagnostics. Слишком короткое окно может потерять сезонную информацию. Поддержка гипотезы требует pooled gain и согласованности по фолдам и маршрутам, а не улучшения одного периода.

In [2]:
names = [fold['name'] for fold in result['folds']]
summary_rows = []
train_rows = []
for window in result['windows']:
    folds = result['fold_results'][window]
    summary = result['summaries'][window]
    comparison = result['comparisons_vs_full'].get(window, {})
    summary_rows.append({'window': window, **{name: fold['wape'] for name, fold in zip(names, folds)},
                         'pooled_wape': summary['pooled_wape'],
                         'delta_vs_full': comparison.get('delta_pooled_wape', 0.0),
                         'pooled_absolute_error': summary['pooled_absolute_error'],
                         'wape_score': summary['wape_score'],
                         'mean_fold_wape': summary['mean_wape'],
                         'median_fold_wape': summary['median_wape'],
                         'fold_wins': comparison.get('fold_wins', 0),
                         'fold_losses': comparison.get('fold_losses', 0),
                         'fit_seconds': summary['train_seconds'],
                         'predict_seconds': summary['inference_seconds']})
    for name, fold in zip(names, folds):
        train_rows.append({'window': window, 'fold': name,
                           **{key: fold[key] for key in ('first_train_date', 'last_train_date',
                              'train_rows', 'min_weight', 'mean_weight', 'max_weight',
                              'effective_weight_sum', 'train_seconds', 'inference_seconds')}})
display(pd.DataFrame(summary_rows).round(6))
display(pd.DataFrame(train_rows).round(4))

,window,jan_apr_to_may_jun,jan_jun_to_jul_aug,jan_aug_to_sep_oct,pooled_wape,delta_vs_full,pooled_absolute_error,wape_score,mean_fold_wape,median_fold_wape,fold_wins,fold_losses,fit_seconds,predict_seconds
0,FULL,0.156928,0.198218,0.118587,0.155601,0.000000,5450387,0.844399,0.157911,0.156928,0,0,30.999298,0.955307
1,8m,0.156928,0.198218,0.118587,0.155601,0.000000,5450387,0.844399,0.157911,0.156928,0,0,30.999298,0.955307
2,6m,0.156928,0.198218,0.120640,0.156348,0.000747,5476567,0.843652,0.158595,0.156928,0,1,27.323108,0.958762
3,4m,0.156928,0.194249,0.131488,0.159084,0.003483,5572392,0.840916,0.160888,0.156928,1,1,21.784985,0.958932
4,2m,0.213561,0.190247,0.237487,0.215139,0.059539,7535908,0.784861,0.213765,0.213561,1,2,10.615550,0.948508


,window,fold,first_train_date,last_train_date,train_rows,min_weight,mean_weight,max_weight,effective_weight_sum,train_seconds,inference_seconds
0,FULL,jan_apr_to_may_jun,2025-01-01,2025-04-30,28800,0.6324,0.8023,1.0,23106.9310,7.6552,0.3226
1,FULL,jan_jun_to_jul_aug,2025-01-01,2025-06-30,43440,0.5000,0.7215,1.0,31342.2514,9.6037,0.3179
2,FULL,jan_aug_to_sep_oct,2025-01-01,2025-08-31,58320,0.3938,0.6507,1.0,37948.0208,13.7404,0.3148
3,8m,jan_apr_to_may_jun,2025-01-01,2025-04-30,28800,0.6324,0.8023,1.0,23106.9310,7.6552,0.3226
4,8m,jan_jun_to_jul_aug,2025-01-01,2025-06-30,43440,0.5000,0.7215,1.0,31342.2514,9.6037,0.3179
5,8m,jan_aug_to_sep_oct,2025-01-01,2025-08-31,58320,0.3938,0.6507,1.0,37948.0208,13.7404,0.3148
6,6m,jan_apr_to_may_jun,2025-01-01,2025-04-30,28800,0.6324,0.8023,1.0,23106.9310,7.6552,0.3226
7,6m,jan_jun_to_jul_aug,2025-01-01,2025-06-30,43440,0.5000,0.7215,1.0,31342.2514,9.6037,0.3179
8,6m,jan_aug_to_sep_oct,2025-03-01,2025-08-31,44160,0.4943,0.7178,1.0,31699.4912,10.0642,0.3183
9,4m,jan_apr_to_may_jun,2025-01-01,2025-04-30,28800,0.6324,0.8023,1.0,23106.9310,7.6552,0.3226


In [3]:
best = result['best_shorter_window']
comparison = result['comparisons_vs_full'][best]
display(pd.DataFrame([{'fold': key, 'delta_wape': value}
                      for key, value in comparison['fold_delta_wape'].items()]).round(6))
display(pd.DataFrame([{'route': key, 'full_wape': result['summaries']['FULL']['per_route'][key]['wape'],
                       'shorter_wape': result['summaries'][best]['per_route'][key]['wape'],
                       'delta_wape': value}
                      for key, value in comparison['route_delta_pooled_wape'].items()]).round(6))

,fold,delta_wape
0,jan_apr_to_may_jun,0.000000
1,jan_jun_to_jul_aug,0.000000
2,jan_aug_to_sep_oct,0.002053


,route,full_wape,shorter_wape,delta_wape
0,1,0.124341,0.124134,-0.000206
1,7,0.273041,0.276641,0.003599
2,11,0.127854,0.128179,0.000325
3,12,0.119222,0.119761,0.000539
4,17,0.117136,0.118531,0.001395
5,25,0.194396,0.194410,0.000014
6,26,0.171733,0.171590,-0.000142
7,28,0.184639,0.185166,0.000528
8,50,0.216200,0.215306,-0.000894


## Вывод

Классификация и воспроизводимые метрики сохраняются в `ml/experiments/results/train_window_experiment.json`. Это историческая проверка; final Jan–Oct fit и submission здесь не выполняются.